# 01 — Train MNIST (MLP / CNN)

Trains an MLP (`784→128→64→10`) or a small CNN on MNIST using cross-entropy + Adam.
The checkpoint is saved to `runs/<RUN_NAME>/model.pt`.

> **Tip:** Enable GPU via *Runtime → Change runtime type → GPU* for faster training.

**Thesis context — WP1:** This notebook covers the training step of the baseline pipeline
for formal robustness verification of neural networks.

In [ ]:
# Install dependencies
!pip install -q torch torchvision numpy pandas pyyaml tqdm ortools

## 1 — Imports & reproducibility

In [ ]:
from __future__ import annotations

import json
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm import tqdm


def set_seed(seed: int, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True, warn_only=True)


print("Imports OK")

## 2 — Data loading (MNIST)

In [ ]:
def get_mnist_datasets(data_dir: str | Path = "data"):
    data_dir = Path(data_dir)
    tfm = transforms.ToTensor()
    train_ds = datasets.MNIST(root=str(data_dir), train=True,  download=True, transform=tfm)
    test_ds  = datasets.MNIST(root=str(data_dir), train=False, download=True, transform=tfm)
    return train_ds, test_ds


def make_loader(dataset, batch_size: int, shuffle: bool, num_workers: int, seed: int):
    gen = torch.Generator()
    gen.manual_seed(int(seed))
    return DataLoader(
        dataset,
        batch_size=int(batch_size),
        shuffle=bool(shuffle),
        num_workers=int(num_workers),
        generator=gen,
        drop_last=False,
        pin_memory=False,
    )


print("Data utilities defined")

## 3 — Model definitions

In [ ]:
class MnistMlp(nn.Module):
    """3-layer ReLU MLP: 784 -> h1 -> h2 -> 10."""

    def __init__(self, in_dim: int = 784, h1: int = 128, h2: int = 64, num_classes: int = 10):
        super().__init__()
        self.in_dim = int(in_dim)
        self.h1 = int(h1)
        self.h2 = int(h2)
        self.num_classes = int(num_classes)
        self.fc1 = nn.Linear(self.in_dim, self.h1)
        self.fc2 = nn.Linear(self.h1, self.h2)
        self.fc3 = nn.Linear(self.h2, self.num_classes)
        self.relu = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim == 4:
            x = x.view(x.shape[0], -1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)

    def linear_layers(self) -> list[nn.Linear]:
        return [self.fc1, self.fc2, self.fc3]


class CnnSmall(nn.Module):
    """Tiny CNN: Conv(1->16)->ReLU->Pool->Conv(16->32)->ReLU->Pool->FC(64)->FC(10)."""

    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.num_classes = int(num_classes)
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64), nn.ReLU(),
            nn.Linear(64, self.num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


print("Models defined: MnistMlp, CnnSmall")

## 4 — Checkpoint I/O

In [ ]:
@dataclass(frozen=True)
class CheckpointMeta:
    model_type: str
    model_kwargs: dict
    run_name: str


def build_model(model_type: str, model_kwargs: dict) -> nn.Module:
    if model_type == "mlp":
        return MnistMlp(**model_kwargs)
    if model_type == "cnn_small":
        return CnnSmall(**model_kwargs)
    raise ValueError(f"Unknown model_type={model_type!r}")


def save_checkpoint(path, model, meta: CheckpointMeta, metrics=None):
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "meta": asdict(meta),
        "model_state_dict": model.state_dict(),
        "metrics": metrics or {},
    }
    torch.save(payload, str(p))
    p.with_suffix(".meta.json").write_text(
        json.dumps(payload["meta"], indent=2) + "\n", encoding="utf-8"
    )


def load_checkpoint(path, map_location="cpu"):
    payload = torch.load(str(path), map_location=map_location)
    raw = payload["meta"]
    meta = CheckpointMeta(
        model_type=str(raw["model_type"]),
        model_kwargs=dict(raw.get("model_kwargs", {})),
        run_name=str(raw.get("run_name", "run")),
    )
    model = build_model(meta.model_type, meta.model_kwargs)
    model.load_state_dict(payload["model_state_dict"])
    return model, meta, payload.get("metrics", {})


print("Checkpoint utilities defined")

## 5 — Configuration

Edit the values below to change hyperparameters.

In [ ]:
# ── Hyperparameters ────────────────────────────────────────────────────────────
SEED         = 1234
DATA_DIR     = "data"
RUN_NAME     = "mlp_mnist"        # directory under runs/
MODEL_TYPE   = "mlp"              # "mlp" or "cnn_small"
BATCH_SIZE   = 128
EPOCHS       = 3
LR           = 1e-3
WEIGHT_DECAY = 0.0
NUM_WORKERS  = 0
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
# ──────────────────────────────────────────────────────────────────────────────

if MODEL_TYPE == "mlp":
    MODEL_KWARGS = {"in_dim": 784, "h1": 128, "h2": 64, "num_classes": 10}
elif MODEL_TYPE == "cnn_small":
    MODEL_KWARGS = {"num_classes": 10}
else:
    raise ValueError(f"Unknown MODEL_TYPE={MODEL_TYPE!r}")

set_seed(SEED)
print(f"Device: {DEVICE} | Model: {MODEL_TYPE} | Epochs: {EPOCHS}")

## 6 — Training

In [ ]:
@torch.no_grad()
def eval_accuracy(model, loader, device):
    model.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(dim=1)
        correct += int((pred == y).sum().item())
        total   += int(y.numel())
    return correct / max(total, 1)


train_ds, test_ds = get_mnist_datasets(DATA_DIR)
train_loader = make_loader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, seed=SEED)
test_loader  = make_loader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, seed=SEED)

model = build_model(MODEL_TYPE, MODEL_KWARGS).to(DEVICE)
opt   = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

metrics = {"epochs": []}
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = correct = total = 0
    start = time.time()
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        logits = model(x)
        loss   = F.cross_entropy(logits, y)
        loss.backward()
        opt.step()
        epoch_loss += float(loss.item()) * int(y.numel())
        correct    += int((logits.argmax(1) == y).sum().item())
        total      += int(y.numel())

    train_loss = epoch_loss / max(total, 1)
    train_acc  = correct   / max(total, 1)
    test_acc   = eval_accuracy(model, test_loader, DEVICE)
    metrics["epochs"].append({
        "epoch": epoch, "train_loss": train_loss,
        "train_acc": train_acc, "test_acc": test_acc,
        "epoch_time_s": time.time() - start,
    })
    print(f"epoch={epoch}  train_loss={train_loss:.4f}  train_acc={train_acc:.4f}  test_acc={test_acc:.4f}")

metrics["total_time_s"] = time.time() - t0
print(f"\nTotal training time: {metrics['total_time_s']:.1f}s")

## 7 — Save checkpoint

In [ ]:
out_dir   = Path("runs") / RUN_NAME
ckpt_path = out_dir / "model.pt"
meta      = CheckpointMeta(model_type=MODEL_TYPE, model_kwargs=MODEL_KWARGS, run_name=RUN_NAME)

save_checkpoint(ckpt_path, model, meta=meta, metrics=metrics)
(out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")

print(f"Checkpoint saved: {ckpt_path}")
print(f"Final test accuracy: {metrics['epochs'][-1]['test_acc']:.4f}")